# 第 11 章习题与解答

> 本章习题围绕 API 调用、输出解析设计、部署流程展开。

## Exercise 11.1（易）

**题目**:用 Python `requests` 库(或 curl)调用一个 OpenAI 兼容的 `/v1/chat/completions` 接口,分别展示流式和非流式两种调用方式。

<details><summary><b>参考答案</b></summary>

In [ ]:
# 需要先启动 serve_openai_api.py:
#   python scripts/serve_openai_api.py --load_from ../model --hidden_size 768
# 以下代码假设服务跑在 localhost:8998

import requests, json

BASE_URL = "http://localhost:8998/v1/chat/completions"
HEADERS = {"Content-Type": "application/json"}
PAYLOAD = {
    "model": "minimind",
    "messages": [{"role": "user", "content": "你好,介绍一下你自己"}],
    "max_tokens": 128,
    "temperature": 0.7,
}

# === 方式 1: 非流式 (stream=False) ===
print("=== 非流式 ===")
payload_no_stream = {**PAYLOAD, "stream": False}
resp = requests.post(BASE_URL, json=payload_no_stream, headers=HEADERS)
data = resp.json()
content = data["choices"][0]["message"]["content"]
print(f"完整回答: {content}")

# === 方式 2: 流式 (stream=True) ===
print("\n=== 流式 ===")
payload_stream = {**PAYLOAD, "stream": True}
resp = requests.post(BASE_URL, json=payload_stream, headers=HEADERS, stream=True)
print("回答: ", end="", flush=True)
for line in resp.iter_lines():
    if not line:
        continue
    line = line.decode("utf-8")
    if line.startswith("data: "):
        chunk_str = line[6:]  # 去掉 "data: " 前缀
        if chunk_str == "[DONE]":
            break
        chunk = json.loads(chunk_str)
        delta = chunk["choices"][0]["delta"]
        if "content" in delta and delta["content"]:
            print(delta["content"], end="", flush=True)
        if "reasoning_content" in delta and delta["reasoning_content"]:
            print(f"[think:{delta['reasoning_content']}]", end="", flush=True)
print()

# 说明:如果没有跑服务,可以用 curl:
# curl -X POST http://localhost:8998/v1/chat/completions \
#   -H "Content-Type: application/json" \
#   -d '{"model":"minimind","messages":[{"role":"user","content":"你好"}],"stream":true}'
print("\n(如果服务未启动,以上请求会失败。用 curl 也可测试。)")

**解析**:

流式调用的三个要点:
1. `stream=True` 在请求 body 中设置,不是 header
2. 响应是 **逐行** 读取的(`iter_lines`),每行以 `data: ` 开头
3. 每个 chunk 是一个独立的 JSON,`delta` 字段包含增量内容

非流式调用更简单:一个 POST 请求,返回完整的 JSON 响应。

> 生产环境推荐流式 —— 用户体验远好于非流式(TTFT 从秒级降到百毫秒级)。

</details>

## Exercise 11.2（中）

**题目**:`<think>` 内容被解析到 `reasoning_content`,`<tool_call>` 被解析到 `tool_calls` —— 为什么不把它们都放在 `content` 字段里?分析这种分离设计的理由。

<details><summary><b>参考答案</b></summary>

In [ ]:
# 用代码演示:如果混在 content 里,会出现什么问题

raw_output = (
    "<think>用户问天气,我需要调用工具</think>\n"
    "我来帮你查一下。\n"
    '<tool_call>{"name":"get_weather","arguments":{"city":"北京"}}</tool_call>'
)

# === 方案 A (错误): 全部塞进 content ===
content_A = raw_output  # 不解析
print("=== 方案 A: 全部塞进 content ===")
print(f"content: {content_A!r}")
print("问题:")
print("  1. 前端要自己解析 <think> 和 <tool_call> 标签")
print("  2. 工具调用框架(如 LangChain)无法识别 tool_calls")
print("  3. 推理过程暴露给用户(可能包含敏感信息)")

# === 方案 B (正确): 分离到不同字段 ===
import re, json
think = re.search(r'<think>(.*?)</think>', raw_output, re.DOTALL).group(1).strip()
tool_match = re.search(r'<tool_call>(.*?)</tool_call>', raw_output, re.DOTALL).group(1).strip()
tool_data = json.loads(tool_match)
content_B = re.sub(r'<think>.*?</think>', '', raw_output, flags=re.DOTALL)
content_B = re.sub(r'<tool_call>.*?</tool_call>', '', content_B, flags=re.DOTALL).strip()

print(f"\n=== 方案 B: 分离字段 ===")
print(f"reasoning_content: {think!r}")
print(f"tool_calls:        {tool_data}")
print(f"content:           {content_B!r}")
print("好处:")
print("  1. 前端可选展示 reasoning_content(折叠/隐藏)")
print("  2. LangChain 直接读 tool_calls 字段,零解析")
print("  3. content 是干净的纯文本")

**解析**:

分离设计的三个理由:

**1. 职责分离(separation of concerns)**

- `content`:给**用户看**的最终回答
- `reasoning_content`:给**开发者/前端**看的推理过程(可选展示)
- `tool_calls`:给**程序**执行的结构化指令

三者的消费者不同,混在一起会迫使每个消费者都自己解析。

**2. 生态兼容**

LangChain、AutoGen 等 Agent 框架直接读 `message.tool_calls` 字段。如果把 tool_call 放在 content 里,框架需要自己做正则解析 —— 容易出错,且每个模型的格式可能不同。

**3. 流式传输友好**

流式时,`content` 的 delta 可以直接拼接到 UI 上。如果把 `<think>` 也放在 content delta 里,前端需要实时判断「现在是在思考还是在回答」,UI 逻辑复杂化。分离字段后,前端只需要分别监听 `delta.reasoning_content` 和 `delta.content`,分别渲染到不同区域。

> 这也是 OpenAI o1/o3 系列采用 `reasoning_content` 字段的原因 —— 推理过程和最终回答有不同的展示需求。

</details>

## Exercise 11.3（难）

**题目**:minimind 模型训练完成后,要部署到 vllm 提供高并发推理服务,需要经过哪些步骤?写出完整的命令和原因。

<details><summary><b>参考答案</b></summary>

In [ ]:
# 展示完整部署流程的脚本 (实际命令在 shell 中执行)
steps = """
=== minimind → vllm 完整部署流程 ===

# ── 步骤 1: 合并 LoRA (如果用了 LoRA 微调) ──
# 修改 convert_model.py 底部,取消注释 convert_merge_base_lora
python scripts/convert_model.py
# 原因: vllm 不支持实时加载 base+lora, 必须先合并
# 输入: full_sft_768.pth + lora_medical_768.pth
# 输出: merge_medical_768.pth (完整权重)

# ── 步骤 2: 转换为 HF transformers 格式 ──
# 修改 convert_model.py 的 torch_path 和 transformers_path
python scripts/convert_model.py
# 原因: vllm 只认 HF transformers 格式, 用 AutoModelForCausalLM 加载
# 转换: MiniMindForCausalLM → Qwen3ForCausalLM (结构兼容)
# 输出: ../minimind-3/ 目录 (config.json + model.bin + tokenizer)

# ── 步骤 3: 验证转换正确性 ──
python -c "
from transformers import AutoModelForCausalLM, AutoTokenizer
model = AutoModelForCausalLM.from_pretrained('../minimind-3')
tokenizer = AutoTokenizer.from_pretrained('../minimind-3')
# 快速测试: 能否正常 forward
import torch
inputs = tokenizer('你好', return_tensors='pt')
out = model.generate(**inputs, max_new_tokens=20)
print(tokenizer.decode(out[0]))
"
# 原因: 确保转换没有破坏模型结构

# ── 步骤 4: 启动 vllm 服务 ──
vllm serve ../minimind-3 \\
    --port 8000 \\
    --served-model-name minimind \\
    --max-model-len 8192
# 原因: vllm 提供 OpenAI 兼容 API, 且有 PagedAttention 等优化
# --served-model-name: API 中的 model 字段
# --max-model-len: 最大上下文长度 (要 <= 训练时的 max_seq_len)

# ── 步骤 5: 测试 API ──
curl -X POST http://localhost:8000/v1/chat/completions \\
    -H "Content-Type: application/json" \\
    -d '{
        "model": "minimind",
        "messages": [{"role": "user", "content": "你好"}],
        "stream": true
    }'
# 原因: vllm 默认兼容 OpenAI API 格式, 可以直接用 curl 测试
"""
print(steps)

**解析**:

部署到 vllm 的核心步骤:

| 步骤 | 命令 | 原因 |
|---|---|---|
| 1. 合并 LoRA | `convert_merge_base_lora` | vllm 不支持实时 LoRA,必须合并 |
| 2. 格式转换 | `convert_torch2transformers` | vllm 只认 HF transformers 格式 |
| 3. 映射到 Qwen3 | `Qwen3Config` + `Qwen3ForCausalLM` | 让 vllm 用 Qwen3 专用优化 kernel |
| 4. 启动服务 | `vllm serve` | PagedAttention + continuous batching |
| 5. 测试 API | `curl` | 验证 OpenAI 兼容性 |

**为什么是这 5 步,不能跳过?**

- **跳过步骤 1**:vllm 加载 base 模型,LoRA 效果丢失
- **跳过步骤 2**:vllm 无法加载 `.pth` 文件(它需要 `config.json` + `pytorch_model.bin`)
- **跳过步骤 3**:用 `trust_remote_code=True` 加载 MiniMind 原始结构 —— 能跑,但没有 Qwen3 的专用 kernel,性能差 3-5 倍
- **跳过步骤 4**:没有推理优化,并发能力差
- **跳过步骤 5**:不知道服务是否正常工作

**MoE 模型的额外步骤**:如果用了 MoE 架构,步骤 2 会触发 expert stacking(`convert_model.py:72-79`),把独立 expert 合并成 stacked tensor。这是 Qwen3MoE 格式的要求。

> vllm 的核心优势:**PagedAttention**(减少 KV cache 内存碎片)和 **continuous batching**(动态拼批)。对于高并发场景,vllm 的吞吐量是 transformers 原生推理的 5-10 倍。

</details>